## 1. Configuration & data loading

In [14]:
import math, time, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import IsolationForest
from sklearn.metrics import roc_auc_score, roc_curve

import torch
import perceval as pcvl
import merlin as ML
try:
    MeasurementStrategy = ML.MeasurementStrategy
    ComputationSpace   = ML.ComputationSpace
except AttributeError:
    from merlin.measurement.strategies import MeasurementStrategy, ComputationSpace

warnings.filterwarnings("ignore")
plt.rcParams["figure.dpi"] = 110

# ---- settings (match qorc_zeroday_v2.py) ----
N_MODES, N_PHOTONS = 24, 3  # 20,3
TEST_SIZE, SEED    = 0.30, 42
N_NOV_COMPS        = 32
FA_BUDGETS         = [0.01, 0.05]


SEEDS     = [42, 7, 123]
ATTACKS   = ["smurf", "teardrop", "satan", "neptune", "back", "portsweep"]

np.random.seed(SEED); torch.manual_seed(SEED)

CSV_CANDIDATES = [
    "NSL_KDD_labeled.csv",
    "../../NSL_KDD_labeled.csv",
    "data/NSL_KDD_labeled.csv",
    "/run/media/minhdtr/Data/quantum/QRC/data/NSL_KDD_labeled.csv",
    r"D:\claude things\QRC\data\NSL_KDD_labeled.csv",
    r"D:\Python\AI\Code\ReservoirComputing\NSL_KDD_labeled.csv",
]
csv_path = next((p for p in CSV_CANDIDATES if Path(p).exists()), None)
assert csv_path, "NSL_KDD_labeled.csv not found — add its path to CSV_CANDIDATES."
print("loading:", csv_path, "| seeds:", SEEDS, "| attacks:", ATTACKS)

loading: /run/media/minhdtr/Data/quantum/QRC/data/NSL_KDD_labeled.csv | seeds: [42, 7, 123] | attacks: ['smurf', 'teardrop', 'satan', 'neptune', 'back', 'portsweep']


In [15]:
df = pd.read_csv(csv_path)
categorical_cols = ["protocol_type", "service", "flag"]
target_col       = "label"
numeric_cols     = [c for c in df.columns if c not in categorical_cols + [target_col]]

y_all       = np.where(df[target_col] == "normal", 0, 1).astype(np.int64)
attack_type = df[target_col].to_numpy()

ohe   = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
X_cat = ohe.fit_transform(df[categorical_cols]).astype(np.float32)
X_num = df[numeric_cols].to_numpy(dtype=np.float32)
X_all = np.concatenate([X_num, X_cat], axis=1)

print(f"{X_all.shape[0]:,} rows, {X_all.shape[1]} features "
      f"({(y_all==0).sum():,} normal / {(y_all==1).sum():,} attack)")
# fingerprint concentration per attack (how much one protocol/service/flag dominates)
fp_conc = {a: df[attack_type==a].groupby(categorical_cols).size().max()/(attack_type==a).sum()
           for a in ATTACKS}
pd.DataFrame({"n":{a:int((attack_type==a).sum()) for a in ATTACKS},
              "fingerprint_conc":fp_conc}).round(2)

125,973 rows, 122 features (67,343 normal / 58,630 attack)


,n,fingerprint_conc
smurf,2646,1.00
teardrop,892,1.00
satan,3633,0.32
neptune,41214,0.31
back,956,0.90
portsweep,2931,0.48


## 2. Photonic reservoir (built once)

In [16]:
def build_quantum_layer(seed):
    """Build one photonic reservoir. Seeded so results are reproducible per seed."""
    np.random.seed(seed)                      # perceval's random_unitary uses global numpy RNG
    U          = pcvl.Matrix.random_unitary(N_MODES)
    precircuit = pcvl.Unitary(U)
    reservoir  = precircuit.copy()
    c_var = pcvl.Circuit(N_MODES)
    for i in range(N_MODES):
        c_var.add(i, pcvl.PS(pcvl.P(f"px{i+1}")))
    qorc_circuit = precircuit // c_var // reservoir
    step = (N_MODES - 1) / (N_PHOTONS - 1)
    input_state = [0] * N_MODES
    for k in range(N_PHOTONS):
        input_state[int(round(k * step))] = 1
    ql = ML.QuantumLayer(
        input_size=N_MODES, circuit=qorc_circuit,
        trainable_parameters=[], input_parameters=["px"],
        input_state=input_state,
        measurement_strategy=MeasurementStrategy.probs(computation_space=ComputationSpace.UNBUNCHED),
        device=torch.device("cpu"))
    ql.eval()
    return ql

n_qfeat = math.comb(N_MODES, N_PHOTONS)
print(f"modes={N_MODES} photons={N_PHOTONS} -> {n_qfeat} quantum features/row")

modes=24 photons=3 -> 2024 quantum features/row


## 3. Shared pipeline

In [17]:
def quantum_features(ql, phases, batch=4096):
    outs = []
    with torch.no_grad():
        for i in range(0, len(phases), batch):
            outs.append(ql(torch.tensor(phases[i:i+batch], dtype=torch.float32)).cpu().numpy())
    return np.concatenate(outs, 0).astype(np.float32)

def to_phase(Z, mu, sd):
    """Smooth tanh squash, no clipping, no wrap-around -> (0, pi)."""
    return ((np.pi/2) * (1 + np.tanh((Z - mu) / (3 * sd)))).astype(np.float32)

def prepare(ql, X_tr, y_tr, X_te, X_extra, seed):
    """scale -> PCA(20) -> phase -> quantum, all fitted on TRAIN."""
    scaler = StandardScaler().fit(X_tr)
    A, B = scaler.transform(X_tr), scaler.transform(X_te)
    C = scaler.transform(X_extra)
    pca = PCA(n_components=N_MODES, random_state=seed).fit(A)
    Za, Zb, Zc = pca.transform(A), pca.transform(B), pca.transform(C)
    mu, sd = Za.mean(0), Za.std(0) + 1e-8
    Qa = quantum_features(ql, to_phase(Za, mu, sd))
    Qb = quantum_features(ql, to_phase(Zb, mu, sd))
    Qc = quantum_features(ql, to_phase(Zc, mu, sd))
    qs = StandardScaler().fit(Qa)
    Qa, Qb, Qc = qs.transform(Qa), qs.transform(Qb), qs.transform(Qc)
    ps = StandardScaler().fit(Za)
    Ra, Rb, Rc = ps.transform(Za), ps.transform(Zb), ps.transform(Zc)   # dials-only
    tr = dict(raw=A, quantum=Qa, red20=Ra)
    te = dict(raw=B, quantum=Qb, red20=Rb)
    ze = dict(raw=C, quantum=Qc, red20=Rc)
    return tr, te, ze

## 4. Novelty detector — learns normal traffic only

In [18]:
class NoveltyDetector:
    """Learns the shape of normal traffic; scores each row by how far it falls
    from it. Mahalanobis (far from centre) + reconstruction error (outside the
    normal subspace). It never sees an attack, so nothing can be 'unseen'."""
    def __init__(self, k=N_NOV_COMPS): self.k = k
    def fit(self, Xn):
        self.mu = Xn.mean(0); Z = Xn - self.mu
        k = min(self.k, Z.shape[1], Z.shape[0]-1)
        self.pca = PCA(n_components=k, svd_solver="randomized", random_state=0).fit(Z)
        P = self.pca.transform(Z); self.sd = P.std(0) + 1e-8
        R = Z - self.pca.inverse_transform(P); r = np.sqrt((R**2).sum(1))
        self.rmu, self.rsd = r.mean(), r.std() + 1e-8
        return self
    def score(self, X):
        Z = X - self.mu; P = self.pca.transform(Z)
        maha  = np.sqrt(((P / self.sd)**2).sum(1))
        R     = Z - self.pca.inverse_transform(P)
        recon = (np.sqrt((R**2).sum(1)) - self.rmu) / self.rsd
        return maha + np.maximum(recon, 0)

def auc(s_normal, s_attack):
    y = np.r_[np.zeros(len(s_normal)), np.ones(len(s_attack))]
    return roc_auc_score(y, np.r_[s_normal, s_attack])

def detection_at_fa(s_normal, s_attack, fa):
    thr = np.quantile(s_normal, 1 - fa)
    return float((s_attack > thr).mean())

## 5. All-attacks sweep — one experiment per attack

`run_one` runs the full zero-day experiment for a single attack under a single
seed, returning the AUC of every detector plus the raw score arrays (used later
for the distribution plots). Detectors:

| tag | family | input | learns from |
|---|---|---|---|
| `SUP_raw` | supervised | classical (122 feat) | normal + other attacks |
| `SUP_q`   | supervised | **quantum** | normal + other attacks |
| `NOV_raw` | novelty | classical (122 feat) | **normal only** |
| `NOV_q`   | novelty | **quantum** | **normal only** |
| `NOV_dials` | novelty | 20-dim PCA | **normal only** |
| `IForest` | novelty | classical | **normal only** |

In [19]:
def run_one(atk, seed, keep_scores=False):
    ql = build_quantum_layer(seed)
    held = (attack_type == atk)
    Xtr, Xte, ytr, yte = train_test_split(
        X_all[~held], y_all[~held], test_size=TEST_SIZE, random_state=seed,
        stratify=y_all[~held])
    tr, te, ze = prepare(ql, Xtr, ytr, Xte, X_all[held], seed)
    nm = (yte == 0); no = (ytr == 0)
    out, scores = {}, {}

    for tag, rep in [("SUP_raw","raw"), ("SUP_q","quantum")]:
        clf = LogisticRegression(max_iter=2000).fit(tr[rep], ytr)
        s_n = clf.predict_proba(te[rep][nm])[:,1]; s_a = clf.predict_proba(ze[rep])[:,1]
        out[tag] = auc(s_n, s_a); scores[tag] = (s_n, s_a)
    for tag, rep in [("NOV_raw","raw"), ("NOV_q","quantum"), ("NOV_dials","red20")]:
        nov = NoveltyDetector().fit(tr[rep][no])
        s_n, s_a = nov.score(te[rep][nm]), nov.score(ze[rep])
        out[tag] = auc(s_n, s_a); scores[tag] = (s_n, s_a)
    iso = IsolationForest(n_estimators=200, random_state=seed, n_jobs=-1).fit(tr["raw"][no])
    s_n, s_a = -iso.score_samples(te["raw"][nm]), -iso.score_samples(ze["raw"])
    out["IForest"] = auc(s_n, s_a); scores["IForest"] = (s_n, s_a)
    return (out, scores) if keep_scores else out

# single-seed pass (SEED) for the headline table + score arrays for plots
DETECTORS = ["SUP_raw","SUP_q","NOV_raw","NOV_q","NOV_dials","IForest"]
t0 = time.time()
single, scores_by_atk = {}, {}
for atk in ATTACKS:
    o, sc = run_one(atk, SEED, keep_scores=True)
    single[atk] = o; scores_by_atk[atk] = sc
    print(f"  {atk:<10s} done ({time.time()-t0:5.1f}s)")
tbl_single = pd.DataFrame(single).T[DETECTORS]
tbl_single.round(3)

  smurf      done ( 66.2s)
  teardrop   done (130.3s)
  satan      done (191.0s)
  neptune    done (252.5s)
  back       done (317.2s)
  portsweep  done (390.3s)


,SUP_raw,SUP_q,NOV_raw,NOV_q,NOV_dials,IForest
smurf,0.973,0.042,0.989,0.989,0.989,0.940
teardrop,1.000,1.000,0.998,0.996,1.000,0.870
satan,0.923,0.963,0.901,0.937,0.949,0.969
neptune,0.950,0.979,0.990,0.974,0.992,0.999
back,0.471,0.599,0.340,0.392,0.230,0.749
portsweep,0.998,0.733,0.974,0.966,0.979,0.997
